# PUF-Chain NDS Reproducibility Notebook

This notebook regenerates the dataset and experiments, then verifies paper-aligned metrics.

Pipeline steps:
1. Run `simulate.py` to regenerate `dataset/` and offline characterization.
2. Run `run_experiments_v2.py` and `run_experiments_v11.py`.
3. Load `output/results.json` and assert all headline values.

In [ ]:
from pathlib import Path
import json
from pprint import pprint

ROOT = Path.cwd()
if not (ROOT / "simulate.py").exists():
    raise FileNotFoundError("Open this notebook from the repository root that contains simulate.py")

DATASET_DIR = ROOT / "dataset"
RESULTS_PATH = ROOT / "output" / "results.json"
print("Repository root:", ROOT)
print("Results path:", RESULTS_PATH)

In [ ]:
import simulate
import run_experiments_v2
import run_experiments_v11

simulate.main()
run_experiments_v2.main()
run_experiments_v11.main()

In [ ]:
results = json.loads(RESULTS_PATH.read_text(encoding="utf-8"))
print("Top-level sections:", sorted(results.keys()))
pprint(results["dataset_summary"])
pprint(results["offline_characterization"])
pprint(results["v2"]["adaptive_ecc"])
pprint(results["v11"])

In [ ]:
def assert_close(actual, expected, tol, label):
    if abs(actual - expected) > tol:
        raise AssertionError(f"{label}: expected {expected}, got {actual}, tolerance {tol}")

In [ ]:
# Dataset + seed
assert results["seed"] == 42
assert results["dataset_summary"]["seed"] == 42
assert results["dataset_summary"]["rows"]["sram"] == 6000
assert results["dataset_summary"]["rows"]["arbiter"] == 6000
assert results["dataset_summary"]["rows"]["hybrid"] == 6000

# Offline characterization
oc = results["offline_characterization"]
assert_close(oc["sram"]["intra_hd_pct"], 3.20, 0.1, "SRAM intra-HD (%)")
assert_close(oc["sram"]["inter_hd_pct"], 46.89, 0.1, "SRAM inter-HD (%)")
assert_close(oc["arbiter"]["intra_hd_pct"], 0.47, 0.1, "Arbiter intra-HD (%)")
assert_close(oc["arbiter"]["inter_hd_pct"], 45.15, 0.1, "Arbiter inter-HD (%)")
assert_close(oc["hybrid"]["intra_hd_pct"], 6.71, 0.1, "Hybrid intra-HD (%)")
assert_close(oc["hybrid"]["inter_hd_pct"], 50.30, 0.1, "Hybrid inter-HD (%)")

# v2 metrics
v2 = results["v2"]
assert_close(v2["xgboost_aggregate"]["mae_bits"], 1.262, 0.02, "XGBoost MAE (bits)")
assert_close(v2["xgboost_aggregate"]["r2"], 0.363, 0.02, "XGBoost R^2")
assert_close(v2["rf_per_bit"]["mean_rmse"], 0.0625, 1e-6, "RF per-bit mean RMSE")
assert_close(v2["rf_per_bit"]["mean_r2"], -0.50, 0.05, "RF per-bit mean R^2")

ae = v2["adaptive_ecc"]
assert ae["cells"] == 420
assert_close(ae["mean_repetition"], 6.56, 0.1, "Adaptive mean repetition")
assert ae["max_repetition"] == 9
assert_close(ae["saving_vs_512_pct"], 18.0, 0.1, "Adaptive saving vs 512 (%)")
assert_close(ae["adaptive_key_recovery"], 0.99636, 5e-4, "Adaptive key recovery")
assert ae["adaptive_key_recovery"] >= 0.99
assert_close(ae["static_rep8_key_recovery_nominal"], 0.99240, 5e-4, "Static rep-8 key recovery (nominal)")

expected_regime = {
    "cold": 0.9955,
    "nominal": 0.9968,
    "warm": 0.9830,
    "attack": 0.6829,
}
for regime, expected in expected_regime.items():
    assert_close(v2["regime_rep8_key_recovery"][regime], expected, 5e-4, f"Rep8 key recovery ({regime})")

iso = v2["isolation_forest"]
assert_close(iso["recall"], 0.9574, 5e-4, "Isolation Forest recall")
assert_close(iso["false_alarm_rate"], 0.0570, 5e-4, "Isolation Forest FAR")
assert_close(iso["precision"], 0.8772, 5e-4, "Isolation Forest precision")
assert_close(iso["roc_auc"], 0.9939, 1e-6, "Isolation Forest ROC-AUC")
assert iso["counts"]["benign_train"] == 4424
assert iso["counts"]["benign_test"] == 1106
assert iso["counts"]["adversarial_test"] == 470

stale = v2["stale_model_sanity"]
assert stale["assertion_passed"] is True
assert stale["min_repetition_when_stale"] >= 8

# v11 metrics
v11 = results["v11"]
assert_close(v11["trimmed_mean_attack_bit_mae"]["fedavg"], 0.0414, 1e-6, "Trimmed-mean baseline MAE")
assert_close(v11["trimmed_mean_attack_bit_mae"]["trimmed_mean"], 0.0022, 1e-6, "Trimmed-mean robust MAE")
assert_close(v11["arrhenius_cold_start"]["gap_closed_pct"], 23.9, 0.1, "Arrhenius cold-start gap closed (%)")
assert v11["pbft_messages"]["n4_f1"] == 29
assert v11["pbft_messages"]["n7_f2"] == 93

print("All paper-aligned checks passed.")

In [ ]:
paper_summary = {
    "adaptive_cells": results["v2"]["adaptive_ecc"]["cells"],
    "adaptive_saving_pct": results["v2"]["adaptive_ecc"]["saving_vs_512_pct"],
    "adaptive_key_recovery": results["v2"]["adaptive_ecc"]["adaptive_key_recovery"],
    "static_rep8_key_recovery_nominal": results["v2"]["adaptive_ecc"]["static_rep8_key_recovery_nominal"],
    "if_recall": results["v2"]["isolation_forest"]["recall"],
    "if_far": results["v2"]["isolation_forest"]["false_alarm_rate"],
    "if_precision": results["v2"]["isolation_forest"]["precision"],
    "if_roc_auc": results["v2"]["isolation_forest"]["roc_auc"],
    "trimmed_mean_fedavg": results["v11"]["trimmed_mean_attack_bit_mae"]["fedavg"],
    "trimmed_mean_robust": results["v11"]["trimmed_mean_attack_bit_mae"]["trimmed_mean"],
    "arrhenius_gap_closed_pct": results["v11"]["arrhenius_cold_start"]["gap_closed_pct"],
    "pbft_n4_f1": results["v11"]["pbft_messages"]["n4_f1"],
    "pbft_n7_f2": results["v11"]["pbft_messages"]["n7_f2"],
}
pprint(paper_summary)